# 01 - Collecting Rocket League data

Connects to the Ballchasing API, walks the EWC 2026 group tree to find
every match, and saves the ones that involve Falcons as a raw CSV.


In [17]:
## Connect to the Ballchasing API



import ballchasing
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()
api = ballchasing.Api(os.getenv("BALLCHASING_API_TOKEN"))

print("Connected successfully")

Connected successfully


In [18]:
# Inspect which stats exist for one player in one replay
sample_replays = list(api.get_replays(group_id="flcn-vs-m8-qszrcck260"))
sample = api.get_replay(sample_replays[0]["id"])
player = sample["blue"]["players"][0]
for section, values in player["stats"].items():
    print(section, list(values.keys()))

core ['shots', 'shots_against', 'goals', 'goals_against', 'saves', 'assists', 'score', 'mvp', 'shooting_percentage']
boost ['bpm', 'bcpm', 'avg_amount', 'amount_collected', 'amount_stolen', 'amount_collected_big', 'amount_stolen_big', 'amount_collected_small', 'amount_stolen_small', 'count_collected_big', 'count_stolen_big', 'count_collected_small', 'count_stolen_small', 'amount_overfill', 'amount_overfill_stolen', 'amount_used_while_supersonic', 'time_zero_boost', 'percent_zero_boost', 'time_full_boost', 'percent_full_boost', 'time_boost_0_25', 'time_boost_25_50', 'time_boost_50_75', 'time_boost_75_100', 'percent_boost_0_25', 'percent_boost_25_50', 'percent_boost_50_75', 'percent_boost_75_100']
movement ['avg_speed', 'total_distance', 'time_supersonic_speed', 'time_boost_speed', 'time_slow_speed', 'time_ground', 'time_low_air', 'time_high_air', 'time_powerslide', 'count_powerslide', 'avg_powerslide_duration', 'avg_speed_percentage', 'percent_slow_speed', 'percent_boost_speed', 'percen

In [19]:
## Find every leaf group
## Sub-groups that actually contain replays - no filtering yet



# Define a function that finds the actual leaf groups (containing replays)
def get_leaf_group_ids(group_id):
    """Recursively find all sub-groups that actually contain replays."""
    # Get the direct sub-groups of this group, as a list
    children = list(api.get_groups(group=group_id))
    
    # If there are no children, this group is a leaf (it holds actual replays)
    if not children:
        return [group_id]
    
    # If there are sub-groups, go one level deeper into each of them
    # keep going until we reach groups with no children (leaves)
    leaf_ids = []
    for child in children:
        leaf_ids.extend(get_leaf_group_ids(child["id"]))
    return leaf_ids

# The top-level groups we start the search from
top_level_groups = ["group-b-rxo17twtow", "playoffs-98hmgz7iw3"]

# Call the function on each top-level group, and collect all leaf IDs into one list
all_leaf_groups = []
for g in top_level_groups:
    all_leaf_groups.extend(get_leaf_group_ids(g))

print(f"Found {len(all_leaf_groups)} leaf groups:", all_leaf_groups)

Found 18 leaf groups: ['r8-vs-kc-63bxo17vrc', 'ml-vs-m8-srokqxhhxh', '5f-vs-r8-lvatfibbq8', 'ml-vs-wc-lfl1pjqcvf', 'kc-vs-ssg-tnldfnlr28', 'flcn-vs-m8-qszrcck260', 'ml-vs-ssg-raq8yhb9wq', '5f-vs-m8-heakvoll7e', 'r8-vs-flcn-4jeuhpkpdy', 'kc-vs-wc-vslxjmanu6', 'nip-vs-kc-t8glcxjcwi', 'grand-finals-tams2b1l9y', 'kc-vs-flcn-1fbqk0yar1', 'nip-vs-ssg-0bbxbw9ji7', 'nip-vs-m8-f729dy019a', 'ssg-vs-sr-lk8rnnyd12', 'flcn-vs-tm-37y0c4e0wi', 'vit-vs-kc-43g9cg57zm']


In [20]:
## Fetch match details, keep only Falcons matches

# Collect all the rows that belong to Falcons players
all_player_rows = []

# For each leaf group ID (from the previous step), ask the API for its replays
for leaf_group_id in all_leaf_groups:
    replays = api.get_replays(group_id=leaf_group_id)
    
    # Each item here is just a short summary (mainly the replay's ID)
    for replay_summary in replays:
        replay_id = replay_summary["id"]
        # Request the full details for this specific replay
        full_replay = api.get_replay(replay_id)
        
        # Rocket League always splits teams into "blue" and "orange" (the game's default team colors)
        team_names = [
            full_replay.get("blue", {}).get("name", ""),
            full_replay.get("orange", {}).get("name", "")
        ]
        
        # Skip matches that don't involve Falcons
        if not any("flcn" in name.lower() for name in team_names):
            continue
        
        # For matches that do involve Falcons, extract each player's stats
        for team_color in ["blue", "orange"]:
            team = full_replay.get(team_color, {})
            for player in team.get("players", []):
                all_player_rows.append({
                    "source_group": leaf_group_id,
                    "replay_id": replay_id,
                    "match_date": full_replay.get("date"),
                    "team_name": team.get("name"),
                    "team_color": team_color,
                    "player_name": player.get("name"),
                    "goals": player["stats"]["core"].get("goals"),
                    "assists": player["stats"]["core"].get("assists"),
                    "saves": player["stats"]["core"].get("saves"),
                    "shots": player["stats"]["core"].get("shots"),
                    "boost_bpm": player["stats"]["boost"].get("bpm"),
                    "avg_speed": player["stats"]["movement"].get("avg_speed"),
                    "boost_stolen": player["stats"]["boost"].get("amount_stolen"),
                    "pct_zero_boost": player["stats"]["boost"].get("percent_zero_boost"),
                    "pct_supersonic": player["stats"]["movement"].get("percent_supersonic_speed"),
                    "pct_defensive_third": player["stats"]["positioning"].get("percent_defensive_third"),
                    "pct_offensive_third": player["stats"]["positioning"].get("percent_offensive_third"),
                    "dist_to_mates": player["stats"]["positioning"].get("avg_distance_to_mates"),
                    "demos_inflicted": player["stats"]["demo"].get("inflicted"),
                    "demos_taken": player["stats"]["demo"].get("taken"),
                })

# Just a check — print how many rows were collected in total
print(f"Collected {len(all_player_rows)} player-match rows")

Collected 138 player-match rows


In [21]:
## Save the collected data as a CSV file



# Store the records we built above in a variable called df
df = pd.DataFrame(all_player_rows)

# Convert it into a CSV file, saved at this path
df.to_csv("../data/raw/rocket_league/ewc2026_falcons_matches.csv", index=False)

# Print a confirmation message with the number of rows saved
print(f"Saved {len(df)} rows")

# Show the first few rows to confirm the conversion actually worked
df.head()

Saved 138 rows


,source_group,replay_id,match_date,team_name,team_color,player_name,goals,assists,saves,shots,boost_bpm,avg_speed,boost_stolen,pct_zero_boost,pct_supersonic,pct_defensive_third,pct_offensive_third,dist_to_mates,demos_inflicted,demos_taken
0,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13T13:47:29+02:00,FLCN,blue,dralii,2,0,1,5,477,1530,764,11.047958,17.110874,46.354210,26.798140,3837,1,2
1,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13T13:47:29+02:00,FLCN,blue,Rw9,1,1,2,4,459,1699,434,10.046940,24.956396,46.298060,18.309399,3734,3,1
2,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13T13:47:29+02:00,FLCN,blue,Kiileerrz,0,1,0,5,409,1524,586,6.732835,12.711911,43.201145,22.106740,3657,0,0
3,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13T13:47:29+02:00,M8,orange,Oski,1,1,3,1,411,1457,288,15.928627,9.358281,58.938705,16.353909,3738,1,1
4,flcn-vs-m8-qszrcck260,31848ff0-b4d6-448e-9bf8-c402182b06e0,2026-08-13T13:47:29+02:00,M8,orange,nass,0,0,4,2,409,1457,232,14.972853,12.864300,53.259170,12.306739,3764,0,0
